# Day 4 Homework — Evaluate Three Real-Model FAQ Answers

**Course:** LLM Application Engineering · SDA-AIE-213  
**Instructor:** Meaad Al-Marri  
**Assignment:** Extra homework, Day 4 (Assignment B: small evaluation)  
**Mode:** `real_local_model`. FLAN-T5 small runs **locally on CPU** inside Colab. No API key, no paid service, `external_inference_api_calls = 0`.

**Goal:** generate answers for three synthetic paraphrases of the same FAQ, record **real tokenizer token counts** and **measured inference time**, run a simple keyword indicator, and then **manually review the meaning** of every answer.

> This is a measurement exercise on 3 cases, not a benchmark. Keep this notebook **separate** from the core `murshid.ipynb` runtime.

## Step 0 — Setup (run first)

This step installs pinned versions, and keeps Colab's own PyTorch if it is already there. If Colab asks for a restart, restart the runtime and then use **Runtime → Run all**.

In [12]:
import importlib.util, subprocess, sys
for pkg in ['torch', 'transformers', 'sentencepiece']:
    if importlib.util.find_spec(pkg) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

import platform, torch, transformers
print('python      :', platform.python_version())
print('torch       :', torch.__version__)
print('transformers:', transformers.__version__)
print('device      : cpu')

python      : 3.13.18
torch       : 2.14.1+cpu
transformers: 5.22.1
device      : cpu


## Step 1 — The synthetic test set and what a correct answer needs

There is **one** synthetic context, and three paraphrased questions about it.

| ID | Question | What a correct answer must convey |
|---|---|---|
| E01 | How can I check my application status? | Use the application's **reference number** |
| E02 | What do I need to check my application status? | The **reference number** |
| E03 | How can I find out the status of my application? | Use the **reference number** (different wording, same need) |

A correct answer must **not** add facts that are missing from the context, such as a website, a phone number, an ID card or a waiting time.

In [13]:
CONTEXT = 'You can check your application status using its reference number.'

CASES = [
    ('E01', 'How can I check my application status?'),
    ('E02', 'What do I need to check my application status?'),
    ('E03', 'How can I find out the status of my application?'),
]

MODEL_ID = 'google/flan-t5-small'
REVISION = '0fc9ddf78a1e988dac52e2dac162b0ede4fd74ab'          # pinned for reproducibility
GENERATION = {'do_sample': False, 'num_beams': 1, 'max_new_tokens': 48}   # greedy decoding

def build_prompt(question):
    return f'Answer the question using the context.\nContext: {CONTEXT}\nQuestion: {question}\nAnswer:'

print(build_prompt(CASES[0][1]))

Answer the question using the context.
Context: You can check your application status using its reference number.
Question: How can I check my application status?
Answer:


## Step 2 — Load the real model (local download, no API)

In [14]:
import time
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

torch.set_num_threads(2)
t0 = time.perf_counter()
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=REVISION, trust_remote_code=False)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_ID, revision=REVISION, trust_remote_code=False,
                                              use_safetensors=True).to('cpu').eval()
print(f'loaded {MODEL_ID} in {time.perf_counter() - t0:.1f} s (includes first download)')

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


loaded google/flan-t5-small in 0.9 s (includes first download)


## Step 3 — Generate, and measure tokens and time

- `input_tokens` is counted by the **real tokenizer**, not by word count.
- `output_tokens_including_special` is the length of the generated sequence, including the special start/end tokens.
- `inference_seconds` is the time of `model.generate()` only, measured with `time.perf_counter()`.
- `reference_keyword_hit` is True when the answer contains both "reference" and "number". This is a **weak lexical indicator, NOT semantic accuracy**.

In [15]:
import json

rows = []
for case_id, question in CASES:
    inputs = tokenizer(build_prompt(question), return_tensors='pt', truncation=True, max_length=256)
    start = time.perf_counter()
    with torch.inference_mode():
        out = model.generate(**inputs, **GENERATION)
    seconds = round(time.perf_counter() - start, 3)
    answer = tokenizer.decode(out[0], skip_special_tokens=True).strip()
    assert answer, 'Empty generation: inspect the runtime'
    hit = 'reference' in answer.lower() and 'number' in answer.lower()
    rows.append({'id': case_id, 'question': question, 'answer': answer,
                 'reference_keyword_hit': hit,
                 'input_tokens': int(inputs['input_ids'].shape[-1]),
                 'output_tokens_including_special': int(out.shape[-1]),
                 'inference_seconds': seconds})

print(f"{'ID':<4} {'answer':<28} {'kw_hit':<7} {'in_tok':>6} {'out_tok':>7} {'sec':>7}")
for r in rows:
    print(f"{r['id']:<4} {r['answer']:<28} {str(r['reference_keyword_hit']):<7} {r['input_tokens']:>6} "
          f"{r['output_tokens_including_special']:>7} {r['inference_seconds']:>7}")
print('\nkeyword hits:', sum(r['reference_keyword_hit'] for r in rows), '/', len(rows))

ID   answer                       kw_hit  in_tok out_tok     sec
E01  Use its reference number     True        34       6    0.68
E02  Reference number             True        36       4   0.274
E03  Use its reference number     True        37       6   0.218

keyword hits: 3 / 3


## Step 4 — Manual (semantic) review of every answer

The keyword check alone isn't enough. For example, *"Don't use the reference number"* would still count as a keyword hit. So I review each answer against two questions:
1. **Correct meaning:** does it tell the user to use the reference number?
2. **Supported:** does it add any fact that is missing from the context?

To help the second question, the cell below flags any answer word that doesn't appear in the context. It's a helper, not a judge. The final verdict is mine, and it's written in the table under the cell.

In [16]:
import re

STOP = {'a', 'an', 'the', 'to', 'your', 'my', 'you', 'can', 'it', 'its', 'use', 'using', 'is', 'of', 'by', 'with'}
context_words = set(re.findall(r'[a-z]+', CONTEXT.lower()))

for r in rows:
    words = set(re.findall(r'[a-z]+', r['answer'].lower())) - STOP
    unsupported = sorted(words - context_words)
    r['words_not_in_context'] = unsupported
    print(f"{r['id']}: {r['answer']!r}")
    print(f"     keyword hit: {r['reference_keyword_hit']} | words not in context: {unsupported or 'none'}")

E01: 'Use its reference number'
     keyword hit: True | words not in context: none
E02: 'Reference number'
     keyword hit: True | words not in context: none
E03: 'Use its reference number'
     keyword hit: True | words not in context: none


### My manual review

*(Written for the answers observed in my run. If a re-run produces different text, update this table to match the actual output; don't change the output to match the table.)*

| ID | Observed answer | Correct meaning? | Unsupported facts? | Verdict |
|---|---|---|---|---|
| E01 | `Use its reference number` | Yes. It tells the user how to check the status | None | ✅ Correct and supported |
| E02 | `Reference number` | Yes. It names exactly what is needed | None | ✅ Correct, but very short (a bare noun phrase, not a full sentence) |
| E03 | `Use its reference number` | Yes. The paraphrase didn't confuse the model | None | ✅ Correct and supported. It also says "its" without saying what "it" is |

**Result:** 3/3 keyword hits and 3/3 correct in my manual review. **But** all three questions are paraphrases of one fact, and the answer is almost a copy of the context, so this is an easy test.

## Step 5 — Save the report

In [17]:
from pathlib import Path

result = {
    'mode': 'real_local_model',
    'model': MODEL_ID,
    'model_revision': REVISION,
    'generation': GENERATION,
    'environment': {'python': platform.python_version(), 'torch': torch.__version__,
                    'transformers': transformers.__version__, 'device': 'cpu'},
    'metric': 'reference+number keyword proxy; NOT semantic accuracy',
    'keyword_hits': sum(r['reference_keyword_hit'] for r in rows),
    'total': len(rows),
    'external_inference_api_calls': 0,
    'billed_api_cost': 0,
    'cases': rows,
    'manual_review': 'Each answer reviewed for meaning and unsupported facts (see notebook table). '
                     'Keyword hits are a lexical indicator only.',
}

# Schema check against the course's expected report
for key in ['mode', 'model', 'generation', 'keyword_hits', 'total', 'external_inference_api_calls', 'cases']:
    assert key in result, key
assert result['total'] == 3 and [c['id'] for c in rows] == ['E01', 'E02', 'E03']

out = Path('murshid_live_outputs'); out.mkdir(exist_ok=True)
(out / 'evaluation_result.json').write_text(json.dumps(result, indent=2, ensure_ascii=False), encoding='utf-8')
print(json.dumps(result, indent=2, ensure_ascii=False))

{
  "mode": "real_local_model",
  "model": "google/flan-t5-small",
  "model_revision": "0fc9ddf78a1e988dac52e2dac162b0ede4fd74ab",
  "generation": {
    "do_sample": false,
    "num_beams": 1,
    "max_new_tokens": 48
  },
  "environment": {
    "python": "3.13.18",
    "torch": "2.14.1+cpu",
    "transformers": "5.22.1",
    "device": "cpu"
  },
  "metric": "reference+number keyword proxy; NOT semantic accuracy",
  "keyword_hits": 3,
  "total": 3,
  "external_inference_api_calls": 0,
  "billed_api_cost": 0,
  "cases": [
    {
      "id": "E01",
      "question": "How can I check my application status?",
      "answer": "Use its reference number",
      "reference_keyword_hit": true,
      "input_tokens": 34,
      "output_tokens_including_special": 6,
      "inference_seconds": 0.68,
      "words_not_in_context": []
    },
    {
      "id": "E02",
      "question": "What do I need to check my application status?",
      "answer": "Reference number",
      "reference_keyword_hit": true

## Interpretation

### What the numbers mean
- **Tokens:** the prompt is about 34–37 tokens, and the answers are only 4–6 tokens. Input tokens change with question length, even though the context is the same.
- **Time:** inference took a fraction of a second per case on CPU. This is **local** execution time on a shared Colab machine, so it changes between runs and doesn't predict API latency.
- **Cost:** `external_inference_api_calls = 0`, and the billed API cost is **0**. Everything ran on Colab's CPU.

### Why the keyword score isn't accuracy
`keyword_hits = 3/3` only shows that two words appeared. It would also count a wrong answer like *"You don't need a reference number"* as a hit. That's why the manual review is part of the evaluation.

### Limitations of this small test set
- **Only 3 cases, all about one fact.** A 3/3 result can't measure general reasoning, Arabic ability, safety or production readiness.
- The answer is almost **copied from the context**, so the test checks *extraction*, not understanding.
- There are no negative cases: no unanswerable question, no off-topic question, and no attempt to get private data.
- English only, while the real Murshid service would need Arabic.

### One improvement
Add **negative and harder cases**, and score them with clear criteria. Examples: a question the context **can't** answer, where the correct output is "I don't know", an Arabic paraphrase, and a question with a misleading word ("Do I need my national ID?"). This would test whether the model avoids **unsupported facts**, not just whether it repeats keywords.